In [1]:

import geopandas as gpd
import pandas as pd
import rioxarray

import matplotlib.pyplot as plt

import xarray as xr



In [2]:
import folium 

folium documentation : https://python-visualization.github.io/folium/latest/

# Acessing Vector Data

 It is recommended to use your own shapefile from Survey of India ,For this tutorial we are using an open git repo data

In [ ]:
url = "https://raw.githubusercontent.com/datta07/INDIAN-SHAPEFILES/master/INDIA/INDIA_STATES.geojson"

gdf = gpd.read_file(url)

In [ ]:
gdf .plot()

In [ ]:
gdf.columns

In [ ]:
gdf.head()

In [ ]:

print("Original CRS:", gdf.crs)
print("Row count:", len(gdf))
print("Unique states:", gdf["STNAME"].nunique())

if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# Center of map (project temporarily to avoid the geographic-CRS centroid warning)
proj_centroid = gdf.to_crs(epsg=3857).geometry.centroid.to_crs(epsg=4326)
center = [proj_centroid.y.mean(), proj_centroid.x.mean()]

m = folium.Map(location=center, zoom_start=5, tiles="CartoDB positron")

folium.GeoJson(
    gdf,
    name="India States",
    style_function=lambda feature: {
        "fillColor": "#3186cc",
        "color": "black",
        "weight": 0.6,
        "fillOpacity": 0.1,
    },
    tooltip=folium.GeoJsonTooltip(fields=["STNAME"]),
    popup=folium.GeoJsonPopup(
        fields=["STNAME", "STNAME_SH", "STCODE11", "State_LGD"]
    ),
).add_to(m)

folium.LayerControl().add_to(m)
output_path = "india_states_map.html"
m.save(output_path)
print(f"Saved to {output_path}")
m

# Extracting Census 2011 data 

Option 1 : extract using https://www.data.gov.in/ API 

In [ ]:
api_key = ''  # swap for your own key
resource_id = ''

url = f"https://api.data.gov.in/resource/{resource_id}?api-key={api_key}&format=csv&limit=1000"
df = pd.read_csv(url)
df.head()

Option 2 : extract from web for example wikipedia https://en.wikipedia.org/wiki/2011_census_of_India

An extracted cleaned data is given

In [ ]:
census=pd.read_excel(path)
census

In [ ]:
census.columns

In [ ]:
census.columns[0]

In [ ]:


# gdf: your GeoDataFrame (STNAME, STNAME_SH, geometry, ...)
# census: your census DataFrame with a column like "State /Union Territory (UT)"

merged = gdf.merge(
    census,
    left_on='STNAME_SH',                              # or STNAME, whichever matches better
    right_on=census.columns[0],
    how="left"
)

unmatched = sorted(merged.loc[merged[census.columns[0]].isna(), 'STNAME'].unique())
print(f"Matched: {merged[census.columns[0]].notna().sum()} / {len(gdf)}")
print("Unmatched:", unmatched)

In [ ]:
merged.columns

In [ ]:
m = folium.Map(location=center, zoom_start=5, tiles="CartoDB positron")

folium.GeoJson(
    merged,
    name="India States",
    style_function=lambda feature: {
        "fillColor": "#3186cc",
        "color": "black",
        "weight": 0.6,
        "fillOpacity": 0.1,
    },
    tooltip=folium.GeoJsonTooltip(fields=["STNAME"]),
    popup=folium.GeoJsonPopup(
    fields=list(census.columns)
    ),
).add_to(m)

folium.LayerControl().add_to(m)
output_path = "india_states_map_census.html"
m.save(output_path)
print(f"Saved to {output_path}")
m